In [3]:
import pandas as pd
import os
import random

class FeedLoader:
    def __init__(self):
        self.feed = []

    def load_isot_feed(self, limit=100):
        fake_path = "news/Fake.csv"
        true_path = "news/True.csv"

        print(f"Checking for data at {fake_path} and {true_path}...")

        if not (os.path.exists(fake_path) and os.path.exists(true_path)):
            print("Error: CSV files not found in 'news/' directory.")
            print("Generating synthetic feed instead.")
            return self._generate_synthetic_feed(limit)

        try:
            # Load Real News (Quality = 1)
            df_real = pd.read_csv(true_path)
            df_real['quality_score'] = 1
            df_real['source_label'] = 'Real_News'

            # Load Fake News (Quality = 0)
            df_fake = pd.read_csv(fake_path)
            df_fake['quality_score'] = 0
            df_fake['source_label'] = 'Fake_News'

            # Sample half limit from each to ensure balance
            half_limit = limit // 2
            df_combined = pd.concat([
                df_real.sample(min(len(df_real), half_limit), random_state=42),
                df_fake.sample(min(len(df_fake), half_limit), random_state=42)
            ]).sample(frac=1, random_state=42).reset_index(drop=True)

            # Format for Simulation
            for idx, row in df_combined.iterrows():
                self.feed.append({
                    "id": f"isot_{idx}",
                    "title": str(row['title']),
                    "content": str(row['text'])[:1000], # Truncate for context window
                    "source": row['source_label'],
                    "quality_score": int(row['quality_score']),
                    "topic_tags": [row.get('subject', 'Politics')]
                })

            print(f"Successfully loaded {len(self.feed)} articles from ISOT.")
            return self.feed

        except Exception as e:
            print(f"Error loading ISOT: {e}")
            return self._generate_synthetic_feed(limit)

    def _generate_synthetic_feed(self, count):
        print("Generating synthetic political headlines...")
        topics = [("Tax Reform", 1), ("Election Fraud Claims", 0), ("Climate Policy", 1), ("Deep State Conspiracy", 0)]
        feed = []
        for i in range(count):
            topic, qual = random.choice(topics)
            feed.append({
                "id": f"syn_{i}",
                "title": f"Breaking news regarding {topic}",
                "content": f"Recent developments in {topic} have sparked debate...",
                "source": "Synthetic",
                "quality_score": qual,
                "topic_tags": ["Politics"]
            })
        return feed

In [4]:
# Initialize Loader
loader = FeedLoader()

# Load Real Data
# Warning: Ensure 'news' folder exists in this directory
real_feed = loader.load_isot_feed(limit=20)

# Verify Output
if real_feed:
    print(f"\nSample Article Title: {real_feed[0]['title']}")
    print(f"Sample Article Quality: {real_feed[0]['quality_score']}")

Checking for data at news/Fake.csv and news/True.csv...
Successfully loaded 20 articles from ISOT.

Sample Article Title: Europe rights watchdog says Turkey's emergency laws go too far
Sample Article Quality: 1


In [5]:
import pandas as pd
import json
import random
import os

# Configuration
FAKE_CSV = "news/Fake.csv"
REAL_CSV = "news/True.csv"
OUTPUT_FEED_FILE = "simulation_feed.json"

TOTAL_ARTICLES = 20
FAKE_RATIO = 0.2  # 20% Fake

def generate_simulation_feed():
    if not os.path.exists(FAKE_CSV) or not os.path.exists(REAL_CSV):
        print(f"Error: Could not find {FAKE_CSV} or {REAL_CSV}.")
        return

    # Calculate counts
    n_fake = int(TOTAL_ARTICLES * FAKE_RATIO)
    n_real = TOTAL_ARTICLES - n_fake

    print(f"Generating feed with {n_real} Real and {n_fake} Fake articles...")

    # Load and Sample
    try:
        df_real = pd.read_csv(REAL_CSV).sample(n_real, random_state=42)
        df_real['quality_score'] = 1
        df_real['source_label'] = 'Real_News_Wire'

        df_fake = pd.read_csv(FAKE_CSV).sample(n_fake, random_state=42)
        df_fake['quality_score'] = 0
        df_fake['source_label'] = 'Alternative_News_Source'

        # Combine
        df_feed = pd.concat([df_real, df_fake]).sample(frac=1, random_state=42).reset_index(drop=True)

        simulation_feed = []
        for idx, row in df_feed.iterrows():
            # Construct entry
            simulation_feed.append({
                "id": f"isot_{idx}",
                "title": str(row['title']),
                "content": str(row['text'])[:1200], # Keep enough context for the LLM
                "source": row['source_label'],
                "quality_score": int(row['quality_score']),
                "topic_tags": [str(row.get('subject', 'Politics'))]
            })

        # Save to JSON
        with open(OUTPUT_FEED_FILE, 'w') as f:
            json.dump(simulation_feed, f, indent=2)
        
        print(f"Success. Saved {len(simulation_feed)} articles to {OUTPUT_FEED_FILE}")

    except Exception as e:
        print(f"Error processing CSVs: {e}")

generate_simulation_feed()

Generating feed with 16 Real and 4 Fake articles...
Success. Saved 20 articles to simulation_feed.json
